# Colab Generation Experiment

VM에서는 Hugging Face generation 모델 로딩이 무거우므로, Colab에서 최종 retrieval 결과를 기반으로 generation만 실행한다.

필요한 입력 파일은 세 가지다.

1. `data/eval/*.csv` 평가 질문 파일들
2. 최종 retrieval prediction JSONL
3. context 보강용 chunk 파일: `chunks.json` 또는 `chunks_v2_*.jsonl`

Vector DB나 embedding index는 필요 없다. generation은 690 DB에서 미리 만들어둔 retrieval prediction을 사용한다.


## 0. Colab GPU 확인


In [ ]:
!nvidia-smi


## 1. Repo clone

Private repo라면 Colab의 GitHub 인증 또는 personal access token 설정이 필요할 수 있다.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = "https://github.com/beomsookim1020/chatbot.git"
BRANCH = "colab-generation"
PROJECT_DIR = Path("/content/chatbot")

if PROJECT_DIR.exists():
    %cd /content/chatbot
    !git fetch origin
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
    %cd /content/chatbot


## 2. 필요한 패키지 설치

전체 `requirements.txt`를 설치하지 않고 generation에 필요한 것만 설치한다. Chroma/FAISS/RAGAS는 여기서 필요 없다.


In [ ]:
!pip install -q -U transformers accelerate safetensors sentencepiece python-dotenv pandas tqdm


## 3. Google Drive 연결 및 입력 파일 복사

Drive에는 아래 구조로 파일을 올려둔다.

```text
MyDrive/chatbot_colab_inputs/
  data/eval/*.csv
  outputs/predictions/80_dense_qdecomp_rrf_diverse250_kure_chroma_690_canonical.jsonl
  data/processed/chunks_v2_690.jsonl
```

`chunks_v2_690.jsonl`을 기본 context 보강 파일로 사용한다. 필요하면 `chunks.json`으로 바꿔도 된다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_INPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_inputs')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_outputs')
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PREDICTION_NAME = '80_dense_qdecomp_rrf_diverse250_kure_chroma_690_canonical.jsonl'
CHUNK_SIDECAR_IN_DRIVE = DRIVE_INPUT_ROOT / 'data/processed/chunks_v2_690.jsonl'

LOCAL_EVAL_DIR = Path('data/eval')
LOCAL_PRED_DIR = Path('outputs/predictions')
LOCAL_CHUNK_DIR = Path('data/processed')
LOCAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_PRED_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CHUNK_DIR.mkdir(parents=True, exist_ok=True)

for csv_path in sorted((DRIVE_INPUT_ROOT / 'data/eval').glob('*.csv')):
    shutil.copy2(csv_path, LOCAL_EVAL_DIR / csv_path.name)

shutil.copy2(DRIVE_INPUT_ROOT / 'outputs/predictions' / PREDICTION_NAME, LOCAL_PRED_DIR / PREDICTION_NAME)
local_chunk_sidecar = LOCAL_CHUNK_DIR / CHUNK_SIDECAR_IN_DRIVE.name
shutil.copy2(CHUNK_SIDECAR_IN_DRIVE, local_chunk_sidecar)

print('eval csv count:', len(list(LOCAL_EVAL_DIR.glob('*.csv'))))
print('prediction:', LOCAL_PRED_DIR / PREDICTION_NAME)
print('chunk sidecar:', local_chunk_sidecar)


## 4. Dry-run으로 프롬프트 확인

모델을 로딩하지 않고 context와 prompt만 만들어본다. 여기서 먼저 오류가 없어야 generation으로 넘어간다.


In [ ]:
PRED_PATH = f'outputs/predictions/{PREDICTION_NAME}'
CHUNK_PATH = str(local_chunk_sidecar)

!python scripts/generate_answer_samples.py \
  --dry-run \
  --limit 3 \
  --predictions {PRED_PATH} \
  --chunk-sidecar {CHUNK_PATH} \
  --output outputs/generation/colab_dry_run_samples.jsonl \
  --review-output outputs/generation/colab_dry_run_review.md


## 5. Generation 실행

처음에는 `RUN_LIMIT=20` 정도로 확인하고, 괜찮으면 `RUN_LIMIT=0`으로 전체 문항을 실행한다.

- `Qwen/Qwen2.5-3B-Instruct`: 빠른 1차 확인용
- `Qwen/Qwen3-4B-Instruct-2507`: 더 무겁지만 성능 확인용 후보


In [ ]:
RUN_LIMIT = 20  # 전체 실행은 0
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_NEW_TOKENS = 384
CONTEXT_MAX_CHARS = 1000

OUTPUT_JSONL = 'outputs/generation/colab_qwen2_5_3b_samples.jsonl'
OUTPUT_REVIEW_MD = 'outputs/generation/colab_qwen2_5_3b_review.md'

!python scripts/generate_answer_samples.py \
  --limit {RUN_LIMIT} \
  --model {MODEL_NAME} \
  --device cuda \
  --torch-dtype float16 \
  --temperature 0.1 \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --context-max-chars {CONTEXT_MAX_CHARS} \
  --max-extra-contexts 4 \
  --max-extra-per-doc 2 \
  --predictions {PRED_PATH} \
  --chunk-sidecar {CHUNK_PATH} \
  --output {OUTPUT_JSONL} \
  --review-output {OUTPUT_REVIEW_MD}


## 6. 사람이 채점하기 쉬운 CSV 만들기

Markdown review도 보관하고, 실제 채점은 CSV에서 하는 편이 편하다.


In [ ]:
import json
import pandas as pd
from pathlib import Path

rows = []
with open(OUTPUT_JSONL, encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        guardrails = item.get('guardrails') or {}
        rows.append({
            'id': item.get('id'),
            'type': item.get('type'),
            'difficulty': item.get('difficulty'),
            'question_type': item.get('question_type'),
            'question': item.get('question'),
            'answer': item.get('answer'),
            'ground_truth_answer': item.get('ground_truth_answer'),
            'ground_truth_docs': item.get('ground_truth_docs'),
            'retrieved_docs': '\n'.join(item.get('retrieved_docs') or []),
            'field_candidates': json.dumps(item.get('field_candidates') or {}, ensure_ascii=False),
            'guardrail_confidence': guardrails.get('confidence'),
            'guardrail_warnings': ', '.join(guardrails.get('warnings') or []),
            'latency_ms': item.get('latency_ms'),
            'correctness': '',
            'evidence_grounded': '',
            'failure_type': '',
            'memo': '',
        })

df = pd.DataFrame(rows)
review_csv = Path('outputs/generation/colab_manual_review.csv')
df.to_csv(review_csv, index=False, encoding='utf-8-sig')
display(df[['id', 'type', 'question_type', 'guardrail_confidence', 'guardrail_warnings', 'question', 'answer', 'ground_truth_answer']].head(20))
print('saved:', review_csv)


## 7. 결과를 Drive에 저장


In [ ]:
for path in [OUTPUT_JSONL, OUTPUT_REVIEW_MD, 'outputs/generation/colab_manual_review.csv']:
    src = Path(path)
    if src.exists():
        shutil.copy2(src, DRIVE_OUTPUT_ROOT / src.name)
        print('copied:', DRIVE_OUTPUT_ROOT / src.name)


## 채점 기준 추천

`failure_type`은 처음에는 아래 값 중 하나로 통일해서 적는다.

- `correct`
- `partial_answer`
- `retrieval_miss`
- `context_missing_answer`
- `llm_extraction_error`
- `wrong_similar_document`
- `wrong_numeric_value`
- `wrong_date_or_period`
- `wrong_field`
- `citation_error`
- `hallucination`
- `format_error`
